# ⚡ Energy Consumption — 3D Animated Galaxy Map
### Real data insights visualized as an interactive moving 3D node graph
---

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✓')

Libraries loaded ✓


## 1. Load & Engineer Features

In [4]:
df = pd.read_excel('Energy Consumption Dataset.xlsx')
df.columns = ['start', 'end', 'consumption']
df['start'] = pd.to_datetime(df['start'])

# Time features
df['year']       = df['start'].dt.year
df['month']      = df['start'].dt.month
df['hour']       = df['start'].dt.hour
df['dayofweek']  = df['start'].dt.dayofweek   # 0=Mon, 6=Sun
df['quarter']    = df['start'].dt.quarter
df['week']       = df['start'].dt.isocalendar().week.astype(int)
df['date']       = df['start'].dt.date
df['month_name'] = df['start'].dt.strftime('%b')
df['day_name']   = df['start'].dt.strftime('%a')
df['is_weekend'] = df['dayofweek'] >= 5
df['season']     = df['month'].map({
    12:'Winter', 1:'Winter', 2:'Winter',
    3:'Spring',  4:'Spring', 5:'Spring',
    6:'Summer',  7:'Summer', 8:'Summer',
    9:'Autumn', 10:'Autumn', 11:'Autumn'
})

# Remove 2015 (only 3 rows)
df = df[df['year'] >= 2016].reset_index(drop=True)

print(f'Records: {len(df):,}')
print(f'Years:   {sorted(df["year"].unique())}')
print(f'Range:   {df["consumption"].min():,} – {df["consumption"].max():,} MWh')
df.head(3)

Records: 52,963
Years:   [2016, 2017, 2018, 2019, 2020, 2021]
Range:   5,341 – 15,105 MWh


,start,end,consumption,year,month,hour,dayofweek,quarter,week,date,month_name,day_name,is_weekend,season
0,2016-01-01 00:00:00,2016-01-01 01:00:00,9722,2016,1,0,4,1,53,2016-01-01,Jan,Fri,False,Winter
1,2016-01-01 01:00:00,2016-01-01 02:00:00,9599,2016,1,1,4,1,53,2016-01-01,Jan,Fri,False,Winter
2,2016-01-01 02:00:00,2016-01-01 03:00:00,9524,2016,1,2,4,1,53,2016-01-01,Jan,Fri,False,Winter


## 2. Quick Statistical Overview

In [5]:
# Aggregations
yearly   = df.groupby('year')['consumption'].agg(['mean','sum','min','max','std']).round(1)
monthly  = df.groupby(['year','month'])['consumption'].mean().reset_index()
hourly   = df.groupby('hour')['consumption'].mean().reset_index()
seasonal = df.groupby('season')['consumption'].mean().reset_index()
by_day   = df.groupby('dayofweek')['consumption'].mean().reset_index()

print('=== Yearly Summary (MWh) ===')
print(yearly.rename(columns={'mean':'Avg','sum':'Total','min':'Min','max':'Max','std':'StdDev'}))
print('\n=== Peak hour:', hourly.loc[hourly['consumption'].idxmax(), 'hour'], ':00')
print('=== Off-peak hour:', hourly.loc[hourly['consumption'].idxmin(), 'hour'], ':00')
print('=== Weekend vs Weekday:')
print(df.groupby('is_weekend')['consumption'].mean().rename({False:'Weekday', True:'Weekend'}))

=== Yearly Summary (MWh) ===
         Avg     Total   Min    Max  StdDev
year                                       
2016  9552.7  84102180  5593  15105  1610.5
2017  9521.9  83345248  5854  14273  1438.7
2018  9791.9  85767200  6348  14062  1633.7
2019  9482.4  86194868  5455  14542  1605.8
2020  8916.4  78464338  5341  12388  1311.8
2021  9669.5  84676090  6450  14267  1678.5

=== Peak hour: 17 :00
=== Off-peak hour: 1 :00
=== Weekend vs Weekday:
is_weekend
Weekday    9664.291778
Weekend    9049.458945
Name: consumption, dtype: float64


## 3. 🌌 Galaxy Node Map — Monthly Consumption Nodes (ANIMATED)

In [6]:
# ─── Build 3D galaxy: each node = (year, month, avg_consumption)
#     X = month, Y = year, Z = avg MWh, size = volume, color = MWh

monthly_3d = df.groupby(['year','month','season'])['consumption'].agg(['mean','sum','std','count']).reset_index()
monthly_3d.columns = ['year','month','season','avg','total','std','count']

# 3D coords: spiral galaxy layout — year as depth, month as angle
angles = (monthly_3d['month'] / 12) * 2 * np.pi
radius = 1 + (monthly_3d['year'] - 2016) * 0.6
monthly_3d['x'] = radius * np.cos(angles)
monthly_3d['y'] = radius * np.sin(angles)
monthly_3d['z'] = (monthly_3d['avg'] - monthly_3d['avg'].min()) / (monthly_3d['avg'].max() - monthly_3d['avg'].min()) * 10
monthly_3d['node_size'] = 4 + (monthly_3d['avg'] - monthly_3d['avg'].min()) / (monthly_3d['avg'].max() - monthly_3d['avg'].min()) * 22
monthly_3d['label'] = monthly_3d.apply(
    lambda r: f"<b>{r['year']} {pd.Timestamp(2000, int(r['month']), 1).strftime('%b')}</b><br>"
              f"Avg: {r['avg']:,.0f} MWh<br>"
              f"Total: {r['total']/1e6:.2f} TWh<br>"
              f"Std Dev: {r['std']:,.0f} MWh<br>"
              f"Season: {r['season']}", axis=1
)

season_color = {'Winter': '#7B2FBE', 'Spring': '#A855F7', 'Summer': '#C084FC', 'Autumn': '#9333EA'}
monthly_3d['color_val'] = monthly_3d['avg']

# Build edge lines connecting same-year nodes in sequence
edge_x, edge_y, edge_z = [], [], []
for yr in sorted(monthly_3d['year'].unique()):
    sub = monthly_3d[monthly_3d['year'] == yr].sort_values('month')
    for i in range(len(sub) - 1):
        r0, r1 = sub.iloc[i], sub.iloc[i+1]
        edge_x += [r0.x, r1.x, None]
        edge_y += [r0.y, r1.y, None]
        edge_z += [r0.z, r1.z, None]

fig = go.Figure()

# Edges (constellation lines)
fig.add_trace(go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode='lines',
    line=dict(color='rgba(147,51,234,0.25)', width=2),
    hoverinfo='skip',
    name='Connections'
))

# Nodes
fig.add_trace(go.Scatter3d(
    x=monthly_3d['x'], y=monthly_3d['y'], z=monthly_3d['z'],
    mode='markers',
    marker=dict(
        size=monthly_3d['node_size'],
        color=monthly_3d['color_val'],
        colorscale=[
            [0.0,  '#EDE9FE'],
            [0.25, '#C4B5FD'],
            [0.5,  '#A855F7'],
            [0.75, '#7C3AED'],
            [1.0,  '#4C1D95'],
        ],
        colorbar=dict(
            title=dict(text='Avg MWh', font=dict(color='#4C1D95')),
            tickfont=dict(color='#4C1D95'),
            x=1.0
        ),
        opacity=0.9,
        line=dict(color='white', width=1)
    ),
    text=monthly_3d['label'],
    hovertemplate='%{text}<extra></extra>',
    name='Month Nodes'
))

# Year label centroids
for yr in sorted(monthly_3d['year'].unique()):
    sub = monthly_3d[monthly_3d['year'] == yr]
    fig.add_trace(go.Scatter3d(
        x=[sub['x'].mean()], y=[sub['y'].mean()], z=[sub['z'].max() + 1.2],
        mode='text',
        text=[str(yr)],
        textfont=dict(size=13, color='#4C1D95', family='Arial Black'),
        hoverinfo='skip',
        showlegend=False
    ))

fig.update_layout(
    title=dict(
        text='⚡ Energy Consumption Galaxy — Monthly Node Map (2016–2021)',
        font=dict(size=20, color='#3B0764', family='Arial'),
        x=0.5
    ),
    scene=dict(
        xaxis=dict(title='← Month angle →', backgroundcolor='#F5F3FF',
                   gridcolor='#DDD6FE', tickfont=dict(color='#6D28D9')),
        yaxis=dict(title='← Month angle →', backgroundcolor='#F5F3FF',
                   gridcolor='#DDD6FE', tickfont=dict(color='#6D28D9')),
        zaxis=dict(title='Consumption (normalised)', backgroundcolor='#EDE9FE',
                   gridcolor='#C4B5FD', tickfont=dict(color='#6D28D9')),
        bgcolor='#FAF9FF',
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.0))
    ),
    paper_bgcolor='#FAF9FF',
    plot_bgcolor='#FAF9FF',
    font=dict(color='#4C1D95'),
    height=680,
    margin=dict(l=0, r=100, t=60, b=0),
    showlegend=False
)

fig.show()
print('💡 Tip: Click + drag to rotate | Scroll to zoom | Hover nodes for insights')

💡 Tip: Click + drag to rotate | Scroll to zoom | Hover nodes for insights


## 4. 🌀 Animated Yearly Spiral — Watch Consumption Orbit Through Time

In [7]:
# Animation: frame per year, showing monthly nodes growing

years = sorted(monthly_3d['year'].unique())
frames = []

for yr in years:
    sub = monthly_3d[monthly_3d['year'] <= yr]

    # edges for this frame
    ex, ey, ez = [], [], []
    for y2 in [y for y in years if y <= yr]:
        s2 = sub[sub['year'] == y2].sort_values('month')
        for i in range(len(s2) - 1):
            r0, r1 = s2.iloc[i], s2.iloc[i+1]
            ex += [r0.x, r1.x, None]
            ey += [r0.y, r1.y, None]
            ez += [r0.z, r1.z, None]

    frames.append(go.Frame(
        data=[
            go.Scatter3d(x=ex, y=ey, z=ez,
                         mode='lines',
                         line=dict(color='rgba(147,51,234,0.25)', width=2),
                         hoverinfo='skip'),
            go.Scatter3d(
                x=sub['x'], y=sub['y'], z=sub['z'],
                mode='markers',
                marker=dict(
                    size=sub['node_size'],
                    color=sub['color_val'],
                    colorscale=[
                        [0.0,'#EDE9FE'],[0.25,'#C4B5FD'],
                        [0.5,'#A855F7'],[0.75,'#7C3AED'],[1.0,'#4C1D95']
                    ],
                    cmin=monthly_3d['color_val'].min(),
                    cmax=monthly_3d['color_val'].max(),
                    opacity=0.9,
                    line=dict(color='white', width=1)
                ),
                text=sub['label'],
                hovertemplate='%{text}<extra></extra>'
            )
        ],
        name=str(yr)
    ))

anim_fig = go.Figure(
    data=frames[0].data,
    frames=frames
)

anim_fig.update_layout(
    title=dict(
        text='🌀 Energy Constellation Growing Over Time — Animated',
        font=dict(size=19, color='#3B0764'),
        x=0.5
    ),
    scene=dict(
        xaxis=dict(title='', backgroundcolor='#F5F3FF', gridcolor='#DDD6FE'),
        yaxis=dict(title='', backgroundcolor='#F5F3FF', gridcolor='#DDD6FE'),
        zaxis=dict(title='Consumption', backgroundcolor='#EDE9FE', gridcolor='#C4B5FD'),
        bgcolor='#FAF9FF',
        camera=dict(eye=dict(x=1.8, y=1.8, z=1.1))
    ),
    paper_bgcolor='#FAF9FF',
    font=dict(color='#4C1D95'),
    height=650,
    margin=dict(l=0, r=0, t=60, b=0),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        y=0.02, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=1200, redraw=True),
                                 fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                   mode='immediate')])
        ],
        bgcolor='#EDE9FE',
        font=dict(color='#4C1D95', size=13),
        bordercolor='#A855F7',
    )],
    sliders=[dict(
        steps=[dict(method='animate', args=[[str(yr)],
                    dict(mode='immediate', frame=dict(duration=0, redraw=True))],
                    label=str(yr)) for yr in years],
        x=0.1, len=0.8, y=0.0,
        currentvalue=dict(prefix='Year: ', font=dict(color='#4C1D95', size=14)),
        font=dict(color='#6D28D9'),
        bgcolor='#EDE9FE',
        activebgcolor='#A855F7',
        bordercolor='#DDD6FE'
    )]
)

anim_fig.show()
print('💡 Hit ▶ Play or drag the slider to watch the galaxy expand year by year')

💡 Hit ▶ Play or drag the slider to watch the galaxy expand year by year


## 5. 🔵 3D Hour × Day × Season Heatmap Globe

In [8]:
# Each node = unique (hour, dayofweek, season) combination
# Position: hour as angle, day as radius, season as Z

hds = df.groupby(['hour','dayofweek','season'])['consumption'].mean().reset_index()
hds.columns = ['hour','dow','season','avg']

season_z = {'Winter': 3, 'Spring': 1, 'Summer': 2, 'Autumn': 0}
season_labels = {0:'Autumn', 1:'Spring', 2:'Summer', 3:'Winter'}
hds['z_val'] = hds['season'].map(season_z)

angle = hds['hour'] / 24 * 2 * np.pi
radius = 0.5 + hds['dow'] * 0.3
hds['x'] = radius * np.cos(angle)
hds['y'] = radius * np.sin(angle)
hds['z'] = hds['z_val'] + (hds['avg'] - hds['avg'].min()) / (hds['avg'].max() - hds['avg'].min()) * 0.8
hds['node_size'] = 5 + (hds['avg'] - hds['avg'].min()) / (hds['avg'].max() - hds['avg'].min()) * 18

day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
hds['label'] = hds.apply(
    lambda r: f"<b>{r['season']} | {day_names[int(r['dow'])]} {int(r['hour']):02d}:00</b><br>"
              f"Avg Consumption: {r['avg']:,.0f} MWh", axis=1
)

fig3 = go.Figure(go.Scatter3d(
    x=hds['x'], y=hds['y'], z=hds['z'],
    mode='markers',
    marker=dict(
        size=hds['node_size'],
        color=hds['avg'],
        colorscale=[
            [0.0,'#F3E8FF'],[0.2,'#D8B4FE'],
            [0.4,'#A855F7'],[0.65,'#7C3AED'],
            [0.85,'#5B21B6'],[1.0,'#2E1065']
        ],
        colorbar=dict(
            title=dict(text='Avg MWh', font=dict(color='#4C1D95')),
            tickfont=dict(color='#4C1D95')
        ),
        opacity=0.85,
        line=dict(color='white', width=0.5)
    ),
    text=hds['label'],
    hovertemplate='%{text}<extra></extra>'
))

# Z-layer labels
for z_pos, lbl in season_labels.items():
    fig3.add_trace(go.Scatter3d(
        x=[0], y=[1.8], z=[z_pos + 0.4],
        mode='text',
        text=[lbl],
        textfont=dict(size=12, color='#6D28D9', family='Arial Black'),
        hoverinfo='skip', showlegend=False
    ))

fig3.update_layout(
    title=dict(
        text='🔵 Hour × Weekday × Season — 3D Consumption Globe',
        font=dict(size=19, color='#3B0764'), x=0.5
    ),
    scene=dict(
        xaxis=dict(title='Hour (cos)', backgroundcolor='#F5F3FF', gridcolor='#DDD6FE',
                   tickfont=dict(color='#6D28D9')),
        yaxis=dict(title='Hour (sin)', backgroundcolor='#F5F3FF', gridcolor='#DDD6FE',
                   tickfont=dict(color='#6D28D9')),
        zaxis=dict(title='Season + consumption', backgroundcolor='#EDE9FE',
                   gridcolor='#C4B5FD', tickfont=dict(color='#6D28D9')),
        bgcolor='#FAF9FF',
        camera=dict(eye=dict(x=2.0, y=2.0, z=1.4))
    ),
    paper_bgcolor='#FAF9FF',
    font=dict(color='#4C1D95'),
    height=660,
    margin=dict(l=0, r=100, t=60, b=0),
    showlegend=False
)

fig3.show()
print('💡 Each ring = one hour of day | Radius = weekday (Mon→Sun outward) | Height = Season')

💡 Each ring = one hour of day | Radius = weekday (Mon→Sun outward) | Height = Season


## 6. 📊 Animated Bar Race — Monthly Total Consumption per Year

In [9]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_race = df.groupby(['year','month_name'])['consumption'].mean().reset_index()
monthly_race['month_name'] = pd.Categorical(monthly_race['month_name'], categories=month_order, ordered=True)
monthly_race = monthly_race.sort_values(['year','month_name'])

purple_shades = [
    '#EDE9FE','#DDD6FE','#C4B5FD','#A78BFA',
    '#8B5CF6','#7C3AED','#6D28D9','#5B21B6',
    '#4C1D95','#3B0764','#2E1065','#1E0544'
]

fig4 = go.Figure()

all_years = sorted(monthly_race['year'].unique())
for i, yr in enumerate(all_years):
    sub = monthly_race[monthly_race['year'] == yr]
    fig4.add_trace(go.Bar(
        x=sub['month_name'],
        y=sub['consumption'],
        name=str(yr),
        marker_color=[purple_shades[j % len(purple_shades)] for j in range(len(sub))],
        visible=(i == 0),
        text=[f"{v:,.0f}" for v in sub['consumption']],
        textposition='outside',
        textfont=dict(color='#4C1D95', size=9),
        hovertemplate='<b>%{x}</b><br>Avg: %{y:,.0f} MWh<extra></extra>'
    ))

# Slider steps
steps = []
for i, yr in enumerate(all_years):
    step = dict(
        method='update',
        label=str(yr),
        args=[
            {'visible': [j == i for j in range(len(all_years))]},
            {'title': f'⚡ Monthly Avg Consumption — {yr}'}
        ]
    )
    steps.append(step)

fig4.update_layout(
    title=dict(text='⚡ Monthly Avg Consumption — 2016', font=dict(size=18, color='#3B0764'), x=0.5),
    sliders=[dict(
        active=0, steps=steps,
        currentvalue=dict(prefix='Year: ', font=dict(color='#4C1D95', size=14)),
        font=dict(color='#6D28D9'),
        bgcolor='#EDE9FE', activebgcolor='#A855F7', bordercolor='#DDD6FE',
        pad=dict(t=50)
    )],
    yaxis=dict(
        title='Avg Consumption (MWh)',
        range=[0, monthly_race['consumption'].max() * 1.15],
        gridcolor='#EDE9FE', tickfont=dict(color='#6D28D9')
    ),
    xaxis=dict(tickfont=dict(color='#6D28D9'), title='Month'),
    paper_bgcolor='#FAF9FF',
    plot_bgcolor='#FAF9FF',
    font=dict(color='#4C1D95'),
    height=520,
    margin=dict(t=60, b=120)
)

fig4.show()
print('💡 Drag the slider to compare monthly patterns across years')

💡 Drag the slider to compare monthly patterns across years


## 7. 🌐 3D Surface — Hour × Month Consumption Landscape

In [10]:
pivot = df.groupby(['hour','month'])['consumption'].mean().unstack()
Z = pivot.values
X, Y = np.meshgrid(np.arange(1, 13), np.arange(0, 24))

fig5 = go.Figure(go.Surface(
    z=Z, x=X, y=Y,
    colorscale=[
        [0.0,  '#EDE9FE'],
        [0.2,  '#C4B5FD'],
        [0.4,  '#A855F7'],
        [0.6,  '#7C3AED'],
        [0.8,  '#5B21B6'],
        [1.0,  '#2E1065'],
    ],
    opacity=0.92,
    contours=dict(
        z=dict(show=True, usecolormap=True, highlightcolor='white', project_z=True)
    ),
    hovertemplate='Month: %{x}<br>Hour: %{y}:00<br>Avg: %{z:,.0f} MWh<extra></extra>',
    showscale=True,
    colorbar=dict(
        title=dict(text='MWh', font=dict(color='#4C1D95')),
        tickfont=dict(color='#4C1D95')
    )
))

fig5.update_layout(
    title=dict(
        text='🌐 Consumption Landscape — Hour of Day × Month of Year',
        font=dict(size=19, color='#3B0764'), x=0.5
    ),
    scene=dict(
        xaxis=dict(
            title='Month',
            tickvals=list(range(1,13)),
            ticktext=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
            backgroundcolor='#F5F3FF', gridcolor='#DDD6FE', tickfont=dict(color='#6D28D9')
        ),
        yaxis=dict(
            title='Hour of Day',
            tickvals=[0,4,8,12,16,20,23],
            ticktext=['00:00','04:00','08:00','12:00','16:00','20:00','23:00'],
            backgroundcolor='#F5F3FF', gridcolor='#DDD6FE', tickfont=dict(color='#6D28D9')
        ),
        zaxis=dict(
            title='Avg MWh',
            backgroundcolor='#EDE9FE', gridcolor='#C4B5FD', tickfont=dict(color='#6D28D9')
        ),
        bgcolor='#FAF9FF',
        camera=dict(eye=dict(x=1.7, y=-1.7, z=1.2))
    ),
    paper_bgcolor='#FAF9FF',
    font=dict(color='#4C1D95'),
    height=660,
    margin=dict(l=0, r=100, t=60, b=0)
)

fig5.show()
print('💡 Peaks = high demand hours/months | Valleys = low usage | Hover for exact MWh')

💡 Peaks = high demand hours/months | Valleys = low usage | Hover for exact MWh


## 8. 📈 Key Insights Dashboard (Static)

In [11]:
fig6 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Avg Consumption by Hour',
        'Avg by Day of Week',
        'Yearly Trend',
        'Seasonal Distribution'
    ],
    vertical_spacing=0.18, horizontal_spacing=0.12
)

purple4 = ['#A855F7','#7C3AED','#5B21B6','#2E1065']

# Hourly
fig6.add_trace(
    go.Scatter(x=hourly['hour'], y=hourly['consumption'],
               mode='lines+markers',
               line=dict(color='#7C3AED', width=2.5),
               marker=dict(size=6, color='#A855F7'),
               fill='tozeroy', fillcolor='rgba(168,85,247,0.15)',
               hovertemplate='%{x}:00 — %{y:,.0f} MWh<extra></extra>'),
    row=1, col=1
)

# Weekday
day_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
fig6.add_trace(
    go.Bar(x=[day_labels[d] for d in by_day['dayofweek']], y=by_day['consumption'],
           marker_color=[
               '#A855F7','#9333EA','#7C3AED','#6D28D9','#5B21B6','#C084FC','#DDD6FE'
           ],
           hovertemplate='%{x}: %{y:,.0f} MWh<extra></extra>'),
    row=1, col=2
)

# Yearly
yr_avg = df.groupby('year')['consumption'].mean().reset_index()
fig6.add_trace(
    go.Scatter(x=yr_avg['year'], y=yr_avg['consumption'],
               mode='lines+markers+text',
               text=[f"{v:,.0f}" for v in yr_avg['consumption']],
               textposition='top center',
               textfont=dict(size=9, color='#4C1D95'),
               line=dict(color='#5B21B6', width=2.5),
               marker=dict(size=9, color='#7C3AED', symbol='diamond'),
               hovertemplate='%{x}: %{y:,.0f} MWh<extra></extra>'),
    row=2, col=1
)

# Seasonal
fig6.add_trace(
    go.Bar(x=seasonal['season'], y=seasonal['consumption'],
           marker_color=['#EDE9FE','#C4B5FD','#A855F7','#5B21B6'],
           marker_line_color='#7C3AED', marker_line_width=1.5,
           hovertemplate='%{x}: %{y:,.0f} MWh<extra></extra>'),
    row=2, col=2
)

fig6.update_layout(
    title=dict(text='📊 Energy Consumption — Key Insights Dashboard',
               font=dict(size=18, color='#3B0764'), x=0.5),
    paper_bgcolor='#FAF9FF',
    plot_bgcolor='#FAF9FF',
    font=dict(color='#4C1D95'),
    height=620,
    showlegend=False
)
fig6.update_xaxes(gridcolor='#EDE9FE')
fig6.update_yaxes(gridcolor='#EDE9FE')

fig6.show()

## 9. 🔍 Key Findings

In [12]:
peak_hour  = hourly.loc[hourly['consumption'].idxmax(), 'hour']
trough_hr  = hourly.loc[hourly['consumption'].idxmin(), 'hour']
peak_month = df.groupby('month')['consumption'].mean().idxmax()
low_month  = df.groupby('month')['consumption'].mean().idxmin()
peak_season= seasonal.loc[seasonal['consumption'].idxmax(), 'season']
wkday_avg  = df[~df['is_weekend']]['consumption'].mean()
wkend_avg  = df[df['is_weekend']]['consumption'].mean()

month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

print('=' * 55)
print('         ⚡ KEY ENERGY INSIGHTS ⚡')
print('=' * 55)
print(f'  Peak demand hour   : {peak_hour}:00 – {peak_hour+1}:00')
print(f'  Lowest demand hour : {trough_hr}:00 – {trough_hr+1}:00')
print(f'  Highest demand month: {month_map[peak_month]}')
print(f'  Lowest demand month : {month_map[low_month]}')
print(f'  Peak season        : {peak_season}')
print(f'  Weekday avg        : {wkday_avg:,.0f} MWh')
print(f'  Weekend avg        : {wkend_avg:,.0f} MWh')
print(f'  Weekend drop       : {(wkday_avg-wkend_avg)/wkday_avg*100:.1f}%')
print(f'  Overall avg        : {df["consumption"].mean():,.0f} MWh')
print(f'  Overall max        : {df["consumption"].max():,.0f} MWh')
print(f'  Overall min        : {df["consumption"].min():,.0f} MWh')
print('=' * 55)

         ⚡ KEY ENERGY INSIGHTS ⚡
  Peak demand hour   : 17:00 – 18:00
  Lowest demand hour : 1:00 – 2:00
  Highest demand month: Jan
  Lowest demand month : Jun
  Peak season        : Winter
  Weekday avg        : 9,664 MWh
  Weekend avg        : 9,049 MWh
  Weekend drop       : 6.4%
  Overall avg        : 9,489 MWh
  Overall max        : 15,105 MWh
  Overall min        : 5,341 MWh
